In [3]:
import awkward as ak
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import pandas as pd
import time
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import uproot

In [4]:
dirpath = "/home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/train"

In [20]:
hit_file_path = Path(dirpath) / 'all_hits.parquet'
track_file_path = Path(dirpath) / 'all_tracks.parquet'
hit_eID_series = pd.read_parquet(hit_file_path, columns=['eventID'])['eventID']
track_eID_series = pd.read_parquet(track_file_path, columns=['eventID'])['eventID']

In [21]:
unique_h_eIDs = hit_eID_series.unique()
unique_h_eIDs.sort()
unique_t_eIDs = track_eID_series.unique()
unique_t_eIDs.sort()

In [22]:
unique_h_eIDs
unique_t_eIDs

array([    0,     1,     2, ..., 37997, 37998, 37999])

In [23]:
print(f'unique hit eventIDS : {len(unique_h_eIDs)}')
print(f'unique track eventIDS : {len(unique_t_eIDs)}')

unique hit eventIDS : 37980
unique track eventIDS : 37978


In [24]:
missing_hit_eIDs = np.setdiff1d(np.arange(38000, dtype=unique_h_eIDs.dtype), unique_h_eIDs)
missing_track_eIDs = np.setdiff1d(np.arange(38000, dtype=unique_t_eIDs.dtype), unique_t_eIDs)

In [25]:
print(missing_hit_eIDs)
print(missing_track_eIDs)

[ 1911  2527  3542  8863  9935 15181 16992 21044 21427 21525 22021 24297
 24517 25041 28484 30269 33852 35032 35305 35460]
[ 1911  2527  3542  5377  8863  9935 15181 16992 21044 21427 21525 22021
 24297 24517 25041 28484 30269 31357 33852 35032 35305 35460]


In [26]:
all_hits = pd.read_parquet(hit_file_path)
all_tracks = pd.read_parquet(track_file_path)

In [27]:
hits_counts = all_hits.groupby('eventID').size()
particles_counts = all_tracks.groupby('eventID').size()

In [28]:
valid_events = hits_counts.index.intersection(particles_counts.index).to_list()

In [29]:
missing_events = np.setdiff1d(np.arange(38000, dtype=unique_t_eIDs.dtype), valid_events)

In [30]:
missing_events

array([ 1911,  2527,  3542,  5377,  8863,  9935, 15181, 16992, 21044,
       21427, 21525, 22021, 24297, 24517, 25041, 28484, 30269, 31357,
       33852, 35032, 35305, 35460])

In [31]:
all_hits[all_hits['eventID']==31357]

,trackID,hitID,det,pdg,x,y,z,time,edep,px,py,pz,eventID
1867976,22628951,11,10,-11,-26.422064,-66.987861,148.098670,8.033792e+06,0.024142,-0.671130,-9.813894,2.522899,31357
1867977,22628951,17,10,-11,-9.642037,-71.351887,160.706917,8.033793e+06,0.178045,-0.942054,-6.677666,-1.306017,31357
1867978,22628951,-20,10,-11,32.941189,-64.363319,153.842550,8.033794e+06,0.013303,-2.405003,6.037357,-0.561282,31357
1867979,22628951,-16,10,-11,32.851932,-64.400290,171.969017,8.033793e+06,0.021009,-3.348055,5.834828,-1.575982,31357
1867980,22628951,12,10,-11,-23.350480,-81.882816,153.785315,8.033792e+06,0.031918,4.112613,-8.507288,3.601010,31357
1867981,22628951,18,10,-11,-6.491918,-84.806381,158.177664,8.033793e+06,0.022348,3.407610,-5.580556,-1.325558,31357
1867982,22628951,14,10,-11,-7.621373,-84.679122,184.305372,8.033793e+06,0.027437,3.462149,-6.004598,-1.139554,31357
1867983,22628951,-19,10,-11,34.328353,-78.115654,154.770051,8.033793e+06,0.020863,1.383889,6.385528,-0.630890,31357
1867984,22628951,-15,10,-11,35.976918,-77.538796,174.888516,8.033793e+06,0.018697,0.829691,6.769230,-1.318828,31357
1867985,22628951,-13,10,-11,35.351228,-77.757735,184.644465,8.033792e+06,0.023515,2.548694,8.926828,3.790602,31357


In [32]:
all_tracks[all_tracks['eventID']==31357]

,trackID,pdg,vx,vy,vz,vt,px,py,pz,eventID


In [63]:
print(f'hit fields : {all_hits.keys()}')
print(f'number of hits = {len(all_hits):e}')
print()
print(f'track fields : {all_tracks.keys()}')
print(f'number of hits = {len(all_tracks):e}')
print()
print(f"number of events listed in training hit DataFrame : {len(all_hits['eventID'].unique())}")
print(f"number of events listed in training track DataFrame : {len(all_tracks['eventID'].unique())}")

hit fields : Index(['trackID', 'hitID', 'det', 'pdg', 'x', 'y', 'z', 'time', 'edep', 'px',
       'py', 'pz', 'eventID'],
      dtype='object')
number of hits = 2.265271e+06

track fields : Index(['trackID', 'pdg', 'vx', 'vy', 'vz', 'vt', 'px', 'py', 'pz', 'eventID'], dtype='object')
number of hits = 2.886130e+05

number of events listed in training hit DataFrame : 37980
number of events listed in training track DataFrame : 37978


In [11]:
dirpath = "/home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/data/train"

hit_file_path = Path(dirpath) / 'all_hits.parquet'
track_file_path = Path(dirpath) / 'all_tracks.parquet'
all_hits = pd.read_parquet(hit_file_path)
all_tracks = pd.read_parquet(track_file_path)

hits_counts = all_hits.groupby('eventID').size()
particles_counts = all_tracks.groupby('eventID').size()

valid_events = sorted(hits_counts.index.intersection(particles_counts.index).to_list())
event_names = [f'event{ID:09d}' for ID in valid_events]
num_events_available = len(valid_events)

num_particles = []

for sample_id in tqdm(valid_events):
    hits = all_hits[all_hits['eventID'] == sample_id]
    particles = all_tracks[all_tracks['eventID'] == sample_id]
    
    num_particles.append(len(particles))

    counts = hits['trackID'].value_counts()
    keep_particle_ids = counts[counts >= 1].index.to_numpy()
    particles = particles[particles['trackID'].isin(keep_particle_ids)]

    particles['particle_idx'] = np.arange(len(particles))
    trackID_to_idx = dict(zip(particles['trackID'].values, particles['particle_idx'].values))

    hits = hits.copy()
    hits['particle_idx'] = hits['trackID'].map(trackID_to_idx)
    hits = hits.dropna(subset=['particle_idx'])
    hits['particle_idx'] = hits['particle_idx'].astype(int)

    hits["on_valid_particle"] = hits["particle_idx"].isin(particles["particle_idx"])

    if len(particles) == 0 or len(hits) == 0:
        print(sample_id)
        
print(f'max number of particles in an event : {max(num_particles)}')

100%|██████████| 37978/37978 [04:59<00:00, 127.00it/s]

max number of particles in an event : 24
